# **Phase 5 - Retrieval Evaluation**

In [ ]:
%pip install -q "transformers>=4.44.2,<4.46.0" sentence-transformers FlagEmbedding

In [9]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES']    = '0'

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

import bm25s
import faiss
from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker

import gc
import random
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
DATA_DIR      = Path('workspace/data/cleaned')
PROCESSED_DIR = Path('workspace/data/processed')

OUTPUT_DIR = Path('workspace/results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## **1. Load Data & Models**

In [28]:
corpus    = pd.read_parquet(DATA_DIR / 'corpus.parquet')
val_split = pd.read_parquet(DATA_DIR / 'val_split.parquet')

corpus_texts = corpus['text'].tolist()
corpus_cids  = corpus['cid'].tolist()

cid2idx  = {c: i for i, c in enumerate(corpus_cids)}
idx2cid  = {i: c for c, i in cid2idx.items()}
cid2text = dict(zip(corpus_cids, corpus_texts))

val_relevant  = val_split['cid'].tolist()
val_questions = val_split['question'].tolist()

RETRIEVAL_K = 30 
FINAL_K     = 10
ALL_RESULTS = {}

In [29]:
bi_encoder = SentenceTransformer(
    'YuITC/vietnamese-embedding-vn-legal',
    model_kwargs={
        'torch_dtype'        : torch.bfloat16,
        'attn_implementation': 'sdpa'
    }
)
bi_encoder.max_seq_length = 512

In [30]:
reranker = FlagReranker(
    'YuITC/bge-reranker-v2-m3-vn-legal', 
    use_fp16=True
)

## **2. Metrics**

In [31]:
def rrf_fusion(ranked_lists, k=20, weights=[0.7, 1.3]):
    if weights is None:
        weights = [1.0] * len(ranked_lists)

    scores = {}
    for ranked, w in zip(ranked_lists, weights):
        for rank, cid in enumerate(ranked):
            if cid not in scores:
                scores[cid] = 0.0
            scores[cid] += w / (rank + 1 + k)

    return sorted(scores.keys(), key=lambda c: scores[c], reverse=True)

In [32]:
def rerank_run(run_cids, questions, cid2text, reranker, top_k=FINAL_K, batch_size=128):
    reranked_runs = []

    all_pairs, query_doc_counts = [], []
    for q, cids in zip(questions, run_cids):
        docs = [cid2text.get(c, "") for c in cids]
        all_pairs.extend([[q, d] for d in docs])
        query_doc_counts.append(len(cids))

    print(f"{len(all_pairs)} query-doc pair")
    
    gc.collect()
    torch.cuda.empty_cache()
    all_scores = reranker.compute_score(all_pairs, batch_size=batch_size, max_length=512)
    
    del all_pairs
    gc.collect()
    start_idx = 0
    for i, cids in enumerate(run_cids):
        num_docs   = query_doc_counts[i]
        end_idx    = start_idx + num_docs
        scores     = all_scores[start_idx:end_idx]
        sorted_idx = np.argsort(scores)[::-1]
        reranked   = [cids[j] for j in sorted_idx]
        reranked_runs.append(reranked[:top_k])
        start_idx = end_idx
        
    return reranked_runs

In [33]:
def recall_at_k(retrieved_cids, relevant_cids, k):
    retrieved = set(retrieved_cids[:k])
    relevant  = set(relevant_cids)
    return len(retrieved & relevant) / len(relevant) if relevant else 0.0

def ndcg_at_k(retrieved_cids, relevant_cids, k):
    retrieved = retrieved_cids[:k]
    relevant  = set(relevant_cids)

    dcg  = sum(1.0 / np.log2(i + 2) for i, cid in enumerate(retrieved) if cid in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i      in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def mrr_at_k(retrieved_cids, relevant_cids, k):
    retrieved = retrieved_cids[:k]
    relevant  = set(relevant_cids)
    
    for i, cid in enumerate(retrieved):
        if cid in relevant: return 1.0 / (i + 1)
    return 0.0

def evaluate(run_cids_list, relevant_cids_list, ks=(1, 3, 5, 10)):
    metrics = {'recall': recall_at_k, 'ndcg': ndcg_at_k, 'mrr': mrr_at_k}
    return {
        f"{name}@{k}": np.mean([
            func(run, rel, k) 
            for run, rel in zip(run_cids_list, relevant_cids_list)
        ])
        for k in ks for name, func in metrics.items()
    }

## **3. Ablation Evaluation**

In [34]:
corpus_tokens = pd.read_parquet(PROCESSED_DIR / 'bm25_corpus_tokens.parquet')['tokens'].tolist()
val_tokens    = pd.read_parquet(PROCESSED_DIR / 'bm25_val_tokens.parquet')['tokens'].tolist()

corpus_tokens = [list(x) for x in corpus_tokens]
val_tokens    = [list(x) for x in val_tokens]

retriever_bm25 = bm25s.BM25(k1=1.5, b=0.75)
retriever_bm25.index(corpus_tokens)

bm25_results, _     = retriever_bm25.retrieve(val_tokens, corpus_cids, RETRIEVAL_K, n_threads=os.cpu_count())
bm25_run            = bm25_results.tolist()
ALL_RESULTS['BM25'] = evaluate(bm25_run, val_relevant)

BM25S Create Vocab:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/10717 [00:00<?, ?it/s]

In [35]:
base_faiss      = faiss.read_index(str(PROCESSED_DIR / 'base_faiss_index.bin'))
base_val_embeds = np.load(PROCESSED_DIR / 'base_val_embeddings.npy', mmap_mode='r')

_, I_base                   = base_faiss.search(base_val_embeds.astype(np.float32), RETRIEVAL_K)

dense_base_run              = [[idx2cid[i] for i in idx_list if i >= 0] for idx_list in I_base]
ALL_RESULTS['Dense (base)'] = evaluate(dense_base_run, val_relevant)

In [36]:
ft_faiss      = faiss.read_index(str(PROCESSED_DIR / 'ft_faiss_index.bin'))
ft_val_embeds = np.load(PROCESSED_DIR / 'ft_val_embeddings.npy', mmap_mode='r')

_, I_ft                      = ft_faiss.search(ft_val_embeds.astype(np.float32), RETRIEVAL_K)

dense_ft_run                 = [[idx2cid[i] for i in idx_list if i >= 0] for idx_list in I_ft]
ALL_RESULTS['Dense (tuned)'] = evaluate(dense_ft_run, val_relevant)

In [38]:
def tune_rrf_hyperparameters(bm25_run, dense_run, val_relevant, questions):
    k_values = [10, 20, 40, 60, 80, 100]
    
    weight_combinations = [
        [1.0, 1.0],
        [0.8, 1.2], [0.6, 1.4], [0.4, 1.6], [0.2, 1.8],
        [1.2, 0.8], [1.4, 0.6], [1.6, 0.4], [1.8, 0.2]
    ]
    
    best_score  = 0
    best_params = {}
    results_log = []
  
    for k, weights in itertools.product(k_values, weight_combinations):
        fusion_run = [
            rrf_fusion([bm25_run[i], dense_run[i]], k=k, weights=weights) 
            for i in range(len(questions))
        ]

        metrics       = evaluate(fusion_run, val_relevant, ks=[10])
        current_score = metrics['ndcg@10']
        
        results_log.append({
            'k'           : k, 
            'weight_bm25' : weights[0], 
            'weight_dense': weights[1], 
            'recall@10'   : metrics['recall@10'],
            'ndcg@10'     : current_score
        })

        if current_score > best_score:
            best_score  = current_score
            best_params = {'k': k, 'weights': weights}

    df_tune_results = pd.DataFrame(results_log).sort_values(by='ndcg@10', ascending=False)
    
    print("\nBest RRF Params:")
    print(f"- k       = {best_params['k']}")
    print(f"- weights = {best_params['weights']} (BM25: {best_params['weights'][0]}, Dense: {best_params['weights'][1]})")
    print(f"- NDCG@10 = {best_score:.4f}")
    
    return best_params, df_tune_results

best_rrf_params, tune_history = tune_rrf_hyperparameters(
    bm25_run     = bm25_run, 
    dense_run    = dense_ft_run,
    val_relevant = val_relevant,
    questions    = val_questions
)

display(tune_history.head(5))


Best RRF Params:
- k       = 10
- weights = [0.2, 1.8] (BM25: 0.2, Dense: 1.8)
- NDCG@10 = 0.7051


,k,weight_bm25,weight_dense,recall@10,ndcg@10
4,10,0.2,1.8,0.874949,0.705117
13,20,0.2,1.8,0.875276,0.703208
3,10,0.4,1.6,0.876691,0.702785
22,40,0.2,1.8,0.876442,0.698258
12,20,0.4,1.6,0.876909,0.694657


In [39]:
best_rrf_params

{'k': 10, 'weights': [0.2, 1.8]}

In [43]:
fusion_tuned_run = [
    rrf_fusion([bm25_run[i], dense_ft_run[i]], k=best_rrf_params['k'], weights=best_rrf_params['weights']) 
    for i in range(len(val_questions))
]
ALL_RESULTS['BM25 + Dense tuned + RRF'] = evaluate(fusion_tuned_run, val_relevant)

In [46]:
%%time
rerank_fusion_tuned_run                          = rerank_run(fusion_tuned_run, val_questions, cid2text, reranker)
ALL_RESULTS['BM25 + Dense tuned + RRF + Rerank'] = evaluate(rerank_fusion_tuned_run, val_relevant)

548687 query-doc pair


Compute Scores: 100%|██████████| 4287/4287 [20:37<00:00,  3.46it/s]


CPU times: user 31min 52s, sys: 30 s, total: 32min 22s
Wall time: 22min 32s


In [ ]:
df_res = pd.DataFrame(ALL_RESULTS)
df_res

,metric,BM25,Dense (base),Dense (tuned),BM25 + Dense tuned + RRF,BM25 + Dense tuned + RRF + Rerank
0,recall@1,0.272884,0.340378,0.511337,0.550745,0.513919
1,ndcg@1,0.284595,0.352151,0.534198,0.569842,0.537091
2,mrr@1,0.284595,0.352151,0.534198,0.569842,0.537091
3,recall@3,0.449286,0.524369,0.734207,0.703726,0.737520
4,ndcg@3,0.379803,0.452632,0.649749,0.646974,0.652899
5,mrr@3,0.364623,0.436254,0.634117,0.638736,0.637305
6,recall@5,0.524198,0.601645,0.803459,0.760769,0.805931
7,ndcg@5,0.411195,0.485019,0.679083,0.671104,0.681950
8,mrr@5,0.382217,0.454281,0.649509,0.651916,0.652627
9,recall@10,0.614864,0.695204,0.871046,0.837843,0.874949


In [53]:
df_res.to_csv(OUTPUT_DIR / 'eval_results.csv')